# Fine-tune Gemma locally with QLoRA

This notebook fine-tunes a **Gemma** model on your local machine using:

- [Hugging Face Transformers](https://huggingface.co/docs/transformers) + [PEFT](https://huggingface.co/docs/peft) (LoRA adapters)
- [TRL](https://huggingface.co/docs/trl) `SFTTrainer` for supervised fine-tuning
- **4-bit QLoRA** to reduce VRAM usage (works on most consumer GPUs)

## Before you start

1. **Hugging Face account** — Gemma weights are gated. Accept the license on the model page, then create an [access token](https://huggingface.co/settings/tokens).
2. **GPU** — CUDA is recommended. An 8 GB GPU can usually run `google/gemma-2-2b-it` with QLoRA.
3. **Install dependencies** (from the repo root):

```bash
uv sync
uv run jupyter notebook
```

4. Set `HF_TOKEN` in your environment, or run the login cell below.

In [ ]:
import os
import torch
from huggingface_hub import login

# Optional: authenticate if HF_TOKEN is not already set in the environment
if not os.getenv("HF_TOKEN") and not os.getenv("HUGGING_FACE_HUB_TOKEN"):
    login()  # prompts for token in the notebook UI

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## Configuration

Adjust these values for your hardware and task. Smaller models need less VRAM; increase `max_seq_length` only if you have headroom.

In [ ]:
from pathlib import Path

# Model — accept the license at https://huggingface.co/google/gemma-2-2b-it
MODEL_ID = "google/gemma-2-2b-it"

# Public instruction dataset (swap for your own JSON/CSV later)
DATASET_ID = "trl-lib/Capybara"
DATASET_SPLIT = "train"
MAX_SAMPLES = 500  # lower for quick smoke tests; set None for full dataset

# Training
OUTPUT_DIR = Path("../outputs/gemma-2-2b-it-qlora")
MAX_SEQ_LENGTH = 1024
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_BATCH_SIZE = 1
GRADIENT_ACCUMULATION_STEPS = 8
LEARNING_RATE = 2e-4

# LoRA
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load and format the dataset

`SFTTrainer` expects a `text` column (or a formatting function). Capybara uses multi-turn `messages`; we flatten them into a single training string.

In [ ]:
from datasets import load_dataset

dataset = load_dataset(DATASET_ID, split=DATASET_SPLIT)
if MAX_SAMPLES is not None:
    dataset = dataset.select(range(min(MAX_SAMPLES, len(dataset))))


def format_messages(example: dict) -> dict:
    """Convert chat messages into a single text field for causal LM training."""
    parts: list[str] = []
    for message in example["messages"]:
        role = message["role"]
        content = message["content"].strip()
        parts.append(f"<start_of_turn>{role}\n{content}<end_of_turn>")
    return {"text": "\n".join(parts)}


dataset = dataset.map(format_messages, remove_columns=dataset.column_names)
print(dataset)
print(dataset[0]["text"][:500], "...")

## Load model with 4-bit QLoRA

4-bit quantization keeps most weights frozen in low precision; only LoRA adapter weights are trained in full precision.

In [ ]:
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)

peft_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="epoch",
    bf16=compute_dtype == torch.bfloat16,
    fp16=compute_dtype == torch.float16,
    optim="paged_adamw_8bit",
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field="text",
    packing=False,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

model

## Train

Training time depends on dataset size, GPU, and `MAX_SAMPLES`. Start with a small sample to verify the pipeline.

In [ ]:
train_result = trainer.train()
train_result

## Save LoRA adapters

Only the adapter weights are saved (small on disk). Merge with the base model later if you need a single checkpoint.

In [ ]:
adapter_dir = OUTPUT_DIR / "final_adapter"
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved adapter to {adapter_dir.resolve()}")

## Quick inference test

Load the fine-tuned adapter on top of the quantized base model and generate a response.

In [ ]:
prompt = (
    "<start_of_turn>user\n"
    "Explain what fine-tuning does in one short paragraph.<end_of_turn>\n"
    "<start_of_turn>model\n"
)

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=False))

## Next steps

- **Your own data**: replace `DATASET_ID` with a local JSON/CSV file using `load_dataset("json", data_files="...")`. Each row needs a `text` field or a `messages` list.
- **Smaller GPU**: try `google/gemma-2-2b-it` (current default) or reduce `MAX_SEQ_LENGTH` / `LORA_R`.
- **Larger model**: `google/gemma-2-9b-it` needs ~24 GB VRAM even with QLoRA; consider cloud GPUs.
- **Gemma 3**: swap `MODEL_ID` to `google/gemma-3-1b-pt` and accept its Hugging Face license.
- **Push to Hub**: set `push_to_hub=True` in `SFTConfig` and add `hub_model_id="your-username/gemma-qlora"`.